# Italy OED Mobility Notebook (Goldthorpe Framework)

This notebook operationalizes the **OED triangle** for Italy using your latest INVALSI dataset stack.

- **O (Origin):** socioeconomic gradient proxy (ESCS Q4-Q1 WLE gap, inverted)
- **E (Education):** proficiency distribution advantage proxy
- **D (Destination):** inclusion proxy from inverse implicit dispersion

The objective is a high-efficiency, reproducible diagnostic of mobility structure in recent years.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'Notebooks' else Path.cwd()
PROC = ROOT / 'local_data' / 'processed'
INVALSI = ROOT / 'local_data' / 'INVALSI'

print('Root:', ROOT)
print('Processed dir exists:', PROC.exists())

In [ ]:
oed = pd.read_csv(PROC / 'italy_oed_triangle_dataset.csv')
components = pd.read_csv(PROC / 'italy_oed_components_by_file.csv')

print('OED rows:', len(oed))
display(components)
display(oed)

## Build Analytical OED Space

We keep rows with complete O, E, D shares for triangle visualization and ranking.

In [ ]:
tri = oed.dropna(subset=['O_share', 'E_share', 'D_share']).copy()
tri['dominant_component'] = tri[['O_share', 'E_share', 'D_share']].idxmax(axis=1).str.replace('_share', '', regex=False)
tri['balance_index'] = 1 - tri[['O_share', 'E_share', 'D_share']].std(axis=1)

display(tri[['year', 'O_share', 'E_share', 'D_share', 'dominant_component', 'balance_index']].sort_values('year'))

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
plot_df = tri.melt(id_vars='year', value_vars=['O_share', 'E_share', 'D_share'], var_name='Component', value_name='Share')
plot_df['Component'] = plot_df['Component'].str.replace('_share', '', regex=False)
sns.barplot(data=plot_df, x='year', y='Share', hue='Component', ax=ax)
ax.set_title('OED Composition by Year (Share Form)')
ax.set_ylabel('Share in OED Composite')
ax.set_xlabel('Year')
plt.tight_layout()
plt.show()

In [ ]:
# Optional ternary chart (if plotly is available)
try:
    import plotly.express as px
    fig = px.scatter_ternary(
        tri,
        a='O_share',
        b='E_share',
        c='D_share',
        color='year',
        hover_data=['O_raw_origin', 'E_raw_education', 'D_raw_destination', 'dominant_component'],
        title='Italy OED Triangle: Recent Years'
    )
    fig.update_traces(marker=dict(size=12))
    fig.show()
except Exception as e:
    print('Plotly ternary unavailable:', e)

## Detailed Lens: Education and Origin Proxies in 2024-2025

This section drills into the latest INVALSI release for grade/subject patterns relevant to mobility mechanisms.

In [ ]:
wle_path = INVALSI / 'dati-sottostanti-le-dashboard-di-tableau-del-rapporto-2024-2025-grado-2-grado-5-grado-10__wle_genere_origine_qescs_g02-g05-g10-ms2025.csv'
levels_path = INVALSI / 'dati-sottostanti-le-dashboard-di-tableau-del-rapporto-2024-2025-grado-8-grado-13__livelli-completo_g8_g13-pop-7.csv'
disp_path = INVALSI / 'eccellenza-accademica-e-dispersione-scolastica-implicita-valori-percentuali__report_generale_unito_dispersione_e_eccellenti_agg_2025.csv'

wle = pd.read_csv(wle_path, sep=';', encoding='latin-1')
wle['WLE_ESCS_Q01'] = pd.to_numeric(wle['WLE_ESCS_Q01'].astype(str).str.replace(',', '.', regex=False), errors='coerce')
wle['WLE_ESCS_Q04'] = pd.to_numeric(wle['WLE_ESCS_Q04'].astype(str).str.replace(',', '.', regex=False), errors='coerce')
wle['origin_gap_q4_q1'] = wle['WLE_ESCS_Q04'] - wle['WLE_ESCS_Q01']

origin_by_grade = wle.groupby(['GRADO', 'MATERIA'], as_index=False)['origin_gap_q4_q1'].mean().sort_values('origin_gap_q4_q1', ascending=False)
display(origin_by_grade.head(20))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=origin_by_grade.head(12), y='GRADO', x='origin_gap_q4_q1', hue='MATERIA', ax=ax)
ax.set_title('Origin Penalty Proxy (ESCS Gap Q4-Q1) by Grade and Subject, 2024-2025')
ax.set_xlabel('WLE gap (Q4 - Q1)')
ax.set_ylabel('Grade')
plt.tight_layout()
plt.show()

In [ ]:
levels = pd.read_csv(levels_path, sep=';', encoding='latin-1')
for c in ['LIVELLO_1', 'LIVELLO_2', 'LIVELLO_3', 'LIVELLO_4', 'LIVELLO_5']:
    levels[c] = pd.to_numeric(levels[c].astype(str).str.replace(',', '.', regex=False), errors='coerce')

levels['high_share'] = levels[['LIVELLO_4', 'LIVELLO_5']].sum(axis=1, min_count=1)
levels['low_share'] = levels['LIVELLO_1']
levels['edu_advantage'] = levels['high_share'] - levels['low_share']
edu_by_area = levels.groupby(['RIPARTIZIONE_GEOGRAFICA', 'MATERIA'], as_index=False)['edu_advantage'].mean()
display(edu_by_area.head(20))

In [ ]:
disp = pd.read_csv(disp_path, sep=';', encoding='latin-1')
disp['Pct_dispersione'] = pd.to_numeric(disp['Pct_dispersione'].astype(str).str.replace(',', '.', regex=False), errors='coerce')
disp['Pct_eccellenze'] = pd.to_numeric(disp['Pct_eccellenze'].astype(str).str.replace(',', '.', regex=False), errors='coerce')
disp['year'] = disp['anno'].astype(str).str.extract(r'(20\d{2})-(?:20)?(\d{2})').apply(lambda s: int(s[0]) + 1 if pd.notna(s[0]) else np.nan, axis=1)

disp_recent = disp[disp['year'].isin([2024, 2025])].copy()
trend = disp_recent.groupby(['year'], as_index=False)[['Pct_dispersione', 'Pct_eccellenze']].mean()
display(trend)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
trend_m = trend.melt(id_vars='year', value_vars=['Pct_dispersione', 'Pct_eccellenze'], var_name='Metric', value_name='Value')
sns.lineplot(data=trend_m, x='year', y='Value', hue='Metric', marker='o', ax=ax)
ax.set_title('Destination Proxies from INVALSI (Dispersion vs Eccellenze)')
ax.set_xlabel('Year')
ax.set_ylabel('Percent')
plt.tight_layout()
plt.show()

## Goldthorpe-Oriented Interpretation Summary

Use this block as a concise interpretation layer linked to the OED theoretical construct.

In [ ]:
summary = tri[['year', 'O_share', 'E_share', 'D_share', 'dominant_component', 'balance_index']].sort_values('year').copy()
summary['regime_note'] = np.where(
    summary['dominant_component'] == 'O',
    'Origin-structured regime: social background effects remain strong.',
    np.where(
        summary['dominant_component'] == 'E',
        'Education-led regime: competence distribution dominates.',
        'Destination-led regime: labor-market inclusion pattern dominates.'
    )
)
display(summary)

## Method Limits and Upgrade Path

Current notebook uses high-quality proxies from recent INVALSI releases. For a stricter Goldthorpe OED identification strategy, next upgrades should add:

1. Parent-origin occupational class information (where linkable).
2. Linked post-school destination panels (education-to-employment transitions).
3. Regional harmonization keys to merge INVALSI and ISTAT labor outcomes at a common territorial level.
4. Counterfactual or decomposition models (e.g., origin-controlled destination contrasts).